# Notebook 02 — Payment Analysis

## Brazilian E-Commerce Public Dataset by Olist

### Objetivo executivo

Investigar **como os clientes pagam**, qual é a relação entre **meio de pagamento, ticket e parcelamento**, e quais comportamentos financeiros merecem atenção de gestores, investidores e acionistas.

### Perguntas de negócio

1. Quais meios de pagamento são mais utilizados?
2. Quais meios concentram maior valor financeiro?
3. O ticket médio muda de acordo com o meio de pagamento?
4. Como o parcelamento se distribui?
5. Compras de maior valor utilizam mais parcelas?
6. Qual é o valor financeiro associado a cada faixa de parcelas?
7. Existem diferenças entre quantidade de pedidos e participação financeira?
8. Que conclusões podem ser levadas para a apresentação executiva?

> **Regra central de governança:** `payments` está na granularidade de pagamento, enquanto `orders_base` está na granularidade de pedido. Métricas de pedido devem ser calculadas depois da consolidação dos pagamentos por `order_id`.


## 1. Configuração e carregamento

Este notebook utiliza as camadas preparadas no Notebook 01:

- `orders_base.csv` — uma linha por pedido;
- `payments` — dados brutos de pagamentos, preservando a granularidade original.

Se os arquivos brutos estiverem disponíveis, o notebook também consegue carregá-los diretamente.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

orders_base_path = PROCESSED_DIR / "orders_base.csv"

if not orders_base_path.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {orders_base_path}. "
        "Execute primeiro o Notebook 01."
    )

orders_base = pd.read_csv(
    orders_base_path,
    parse_dates=["order_purchase_timestamp"]
)

payments_path = RAW_DIR / "olist_order_payments_dataset.csv"

if not payments_path.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {payments_path}. "
        "Coloque os dados da Olist em data/raw/."
    )

payments = pd.read_csv(payments_path)

print("orders_base:", orders_base.shape)
print("payments:", payments.shape)


## 2. Inspeção inicial

In [ ]:
display(orders_base.head())
display(payments.head())


In [ ]:
payments.info()


In [ ]:
payment_nulls = (
    payments.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("nulls")
)

payment_nulls["null_pct"] = payment_nulls["nulls"] / len(payments) * 100
payment_nulls


### Observação

Os principais campos utilizados neste notebook são:

- `order_id`: identifica o pedido;
- `payment_sequential`: sequência do pagamento dentro do pedido;
- `payment_type`: meio de pagamento;
- `payment_installments`: número de parcelas;
- `payment_value`: valor daquele registro de pagamento.

Um mesmo pedido pode possuir mais de um registro de pagamento. Portanto, **não devemos somar `payment_value` diretamente após um join com `order_items`**.


## 3. Cobertura e consistência dos pagamentos

In [ ]:
payment_orders = payments["order_id"].nunique()
orders_with_payment = orders_base["order_id"].isin(payments["order_id"]).sum()

coverage = pd.Series({
    "orders_base": orders_base["order_id"].nunique(),
    "orders_with_payment": orders_with_payment,
    "payment_orders": payment_orders,
    "orders_without_payment": orders_base["order_id"].nunique() - orders_with_payment,
    "payment_rows": len(payments),
})

coverage


In [ ]:
payments_per_order = (
    payments.groupby("order_id")
    .size()
    .rename("payment_rows")
)

payments_per_order.describe()


In [ ]:
multiple_payment_orders = (
    payments_per_order
    .gt(1)
    .sum()
)

pd.Series({
    "orders_with_multiple_payment_records": multiple_payment_orders,
    "share_of_payment_orders_with_multiple_records":
        multiple_payment_orders / len(payments_per_order)
})


### Interpretação

A existência de múltiplos registros por pedido pode ocorrer por diferentes combinações de pagamentos. Isso significa que:

- **quantidade de registros em `payments` não representa quantidade de pedidos**;
- `payment_value` representa o valor daquele registro;
- para métricas financeiras no nível do pedido, devemos consolidar `payments` por `order_id`.

Para análises por meio de pagamento, entretanto, manteremos a granularidade original para preservar a composição dos pagamentos.


## 4. Meio de pagamento

In [ ]:
payment_type_summary = (
    payments
    .groupby("payment_type")
    .agg(
        payment_records=("payment_type", "size"),
        orders=("order_id", "nunique"),
        payment_value=("payment_value", "sum"),
        avg_payment_record=("payment_value", "mean"),
        median_payment_record=("payment_value", "median"),
    )
    .sort_values("payment_value", ascending=False)
)

payment_type_summary["share_records"] = (
    payment_type_summary["payment_records"]
    / payment_type_summary["payment_records"].sum()
)

payment_type_summary["share_orders"] = (
    payment_type_summary["orders"]
    / payments["order_id"].nunique()
)

payment_type_summary["share_value"] = (
    payment_type_summary["payment_value"]
    / payment_type_summary["payment_value"].sum()
)

payment_type_summary


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

plot_data = (
    payment_type_summary
    .sort_values("share_value", ascending=True)
)

ax.barh(plot_data.index, plot_data["share_value"] * 100)
ax.set_title("Participação do valor pago por meio de pagamento")
ax.set_xlabel("Participação financeira (%)")
ax.set_ylabel("Meio de pagamento")

plt.tight_layout()
plt.show()


### Insight executivo a procurar

Compare **participação em pedidos** com **participação em valor**.

Se um meio tiver:

> participação financeira > participação em pedidos

isso indica que ele está associado a pagamentos de maior valor relativo.

Se ocorrer o inverso:

> participação financeira < participação em pedidos

o meio tende a concentrar compras de menor valor.

Essa diferença é mais informativa para investidores do que apenas apontar qual meio é o mais popular.


## 5. Ticket médio por meio de pagamento

Para calcular o **ticket por meio de pagamento**, há uma questão metodológica.

Um pedido pode ter mais de um registro de pagamento. Portanto, o `payment_value` de um registro não necessariamente representa o valor total do pedido.

Vamos produzir duas métricas:

1. **Valor médio por registro de pagamento** — útil para entender a operação de pagamentos.
2. **Ticket médio dos pedidos associados ao meio** — útil para comportamento de compra.

Para a segunda métrica, associamos cada pedido aos meios utilizados. Pedidos com múltiplos meios podem aparecer em mais de uma categoria; por isso, essa tabela deve ser interpretada como **ticket dos pedidos que utilizaram o meio**, e não como decomposição exclusiva da receita.


In [ ]:
order_payment_total = (
    payments
    .groupby("order_id", as_index=False)
    .agg(
        order_payment_value=("payment_value", "sum"),
        payment_record_count=("payment_sequential", "count"),
        installment_max=("payment_installments", "max"),
    )
)

order_payment_types = (
    payments[["order_id", "payment_type"]]
    .drop_duplicates()
)

payment_order_level = (
    order_payment_types
    .merge(order_payment_total, on="order_id", how="left", validate="many_to_one")
)

payment_ticket_by_type = (
    payment_order_level
    .groupby("payment_type")
    .agg(
        orders=("order_id", "nunique"),
        avg_order_value=("order_payment_value", "mean"),
        median_order_value=("order_payment_value", "median"),
        total_order_value=("order_payment_value", "sum"),
    )
    .sort_values("avg_order_value", ascending=False)
)

payment_ticket_by_type


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

plot_data = payment_ticket_by_type.sort_values("avg_order_value")

ax.barh(
    plot_data.index,
    plot_data["avg_order_value"]
)

ax.set_title("Ticket médio dos pedidos por meio utilizado")
ax.set_xlabel("Valor médio do pedido")
ax.set_ylabel("Meio de pagamento")

plt.tight_layout()
plt.show()


## 6. Distribuição de parcelas

In [ ]:
installment_summary = (
    payments
    .groupby("payment_installments")
    .agg(
        payment_records=("payment_installments", "size"),
        orders=("order_id", "nunique"),
        payment_value=("payment_value", "sum"),
        avg_payment=("payment_value", "mean"),
        median_payment=("payment_value", "median"),
    )
    .sort_index()
)

installment_summary["share_value"] = (
    installment_summary["payment_value"]
    / installment_summary["payment_value"].sum()
)

installment_summary.head(30)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

plot_data = installment_summary.head(15)

ax.bar(
    plot_data.index.astype(str),
    plot_data["orders"]
)

ax.set_title("Pedidos por quantidade de parcelas — principais faixas")
ax.set_xlabel("Número de parcelas")
ax.set_ylabel("Quantidade de pedidos")

plt.tight_layout()
plt.show()


### Por que limitar o gráfico às principais faixas?

A distribuição de parcelas pode ter uma cauda longa. Mostrar todas as categorias pode prejudicar a leitura executiva.

No relatório final, devemos destacar as faixas que concentram a maior parte dos pedidos e do valor.


## 7. Valor financeiro por número de parcelas

In [ ]:
installment_value = (
    payments
    .groupby("payment_installments")
    .agg(
        orders=("order_id", "nunique"),
        payment_value=("payment_value", "sum"),
        avg_payment=("payment_value", "mean"),
        median_payment=("payment_value", "median"),
    )
    .sort_index()
)

installment_value["share_value"] = (
    installment_value["payment_value"]
    / installment_value["payment_value"].sum()
)

installment_value.head(20)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

plot_data = installment_value.head(15)

ax.bar(
    plot_data.index.astype(str),
    plot_data["share_value"] * 100
)

ax.set_title("Participação financeira por quantidade de parcelas")
ax.set_xlabel("Número de parcelas")
ax.set_ylabel("Participação do valor pago (%)")

plt.tight_layout()
plt.show()


## 8. Parcelamento × valor da compra

Uma hipótese relevante para o negócio é:

> **Compras de maior valor tendem a utilizar mais parcelas?**

Para testá-la, calculamos o valor total pago por pedido e relacionamos esse valor ao número de parcelas.

Como um pedido pode ter mais de um registro de pagamento, utilizaremos o **máximo de parcelas informado no pedido** como indicador simplificado de parcelamento.

> Esta é uma escolha analítica. Em uma análise financeira mais sofisticada, poderíamos estudar cada componente do pagamento separadamente.


In [ ]:
order_payment_behavior = (
    payments
    .groupby("order_id", as_index=False)
    .agg(
        order_payment_value=("payment_value", "sum"),
        max_installments=("payment_installments", "max"),
        payment_types=("payment_type", lambda x: ", ".join(sorted(set(x)))),
        payment_records=("payment_sequential", "count"),
    )
)

order_payment_behavior.describe()


In [ ]:
installment_ticket = (
    order_payment_behavior
    .groupby("max_installments")
    .agg(
        orders=("order_id", "nunique"),
        avg_order_value=("order_payment_value", "mean"),
        median_order_value=("order_payment_value", "median"),
    )
    .sort_index()
)

installment_ticket.head(20)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

plot_data = installment_ticket.head(15)

ax.plot(
    plot_data.index,
    plot_data["avg_order_value"],
    marker="o"
)

ax.set_title("Ticket médio por quantidade máxima de parcelas")
ax.set_xlabel("Número de parcelas")
ax.set_ylabel("Ticket médio")

plt.tight_layout()
plt.show()


### Teste estatístico/quantitativo da hipótese

A correlação não prova causalidade, mas ajuda a avaliar se existe associação monotônica entre parcelas e valor do pedido.



In [ ]:
spearman_corr = order_payment_behavior[
    ["order_payment_value", "max_installments"]
].corr(method="spearman").iloc[0, 1]

pearson_corr = order_payment_behavior[
    ["order_payment_value", "max_installments"]
].corr(method="pearson").iloc[0, 1]

pd.Series({
    "spearman_correlation": spearman_corr,
    "pearson_correlation": pearson_corr
})


### Como interpretar

- correlação positiva: pedidos de maior valor tendem a apresentar mais parcelas;
- correlação próxima de zero: pouca associação linear/monotônica;
- correlação negativa: pedidos de maior valor tendem a utilizar menos parcelas.

**Não interpretar correlação como causalidade.**



## 9. Distribuição do ticket por parcelas

In [ ]:
# Remove extremos apenas para facilitar a visualização.
# A análise principal continua utilizando todos os pedidos.
q99 = order_payment_behavior["order_payment_value"].quantile(0.99)

box_data = order_payment_behavior[
    order_payment_behavior["order_payment_value"] <= q99
    & order_payment_behavior["max_installments"].between(1, 12)
]

fig, ax = plt.subplots(figsize=(12, 6))

sns.boxplot(
    data=box_data,
    x="max_installments",
    y="order_payment_value",
    ax=ax
)

ax.set_title("Distribuição do valor do pedido por parcelas — até P99 e 12 parcelas")
ax.set_xlabel("Número máximo de parcelas")
ax.set_ylabel("Valor do pedido")

plt.tight_layout()
plt.show()


## 10. Cartão de crédito × parcelamento

Como o parcelamento é particularmente relevante para cartão de crédito, vamos separar esse meio.



In [ ]:
credit_card = payments[
    payments["payment_type"].eq("credit_card")
].copy()

credit_installments = (
    credit_card
    .groupby("payment_installments")
    .agg(
        payment_records=("payment_installments", "size"),
        orders=("order_id", "nunique"),
        payment_value=("payment_value", "sum"),
        avg_payment=("payment_value", "mean"),
        median_payment=("payment_value", "median"),
    )
    .sort_index()
)

credit_installments["share_orders"] = (
    credit_installments["orders"]
    / credit_card["order_id"].nunique()
)

credit_installments["share_value"] = (
    credit_installments["payment_value"]
    / credit_card["payment_value"].sum()
)

credit_installments.head(20)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

plot_data = credit_installments.head(15)

ax.bar(
    plot_data.index.astype(str),
    plot_data["share_value"] * 100
)

ax.set_title("Cartão de crédito — participação do valor por parcelas")
ax.set_xlabel("Número de parcelas")
ax.set_ylabel("Participação financeira (%)")

plt.tight_layout()
plt.show()


## 11. Meio de pagamento × faixa de ticket

Para transformar a análise em comportamento, vamos criar faixas de valor por pedido.



In [ ]:
# Integra o meio de pagamento ao valor consolidado do pedido
payment_behavior = (
    payments[["order_id", "payment_type"]]
    .drop_duplicates()
    .merge(
        order_payment_total[["order_id", "order_payment_value"]],
        on="order_id",
        how="left",
        validate="many_to_one"
    )
)

# Faixas de ticket
bins = [-np.inf, 50, 100, 250, 500, 1000, np.inf]
labels = [
    "Até R$ 50",
    "R$ 50–100",
    "R$ 100–250",
    "R$ 250–500",
    "R$ 500–1.000",
    "Acima de R$ 1.000",
]

payment_behavior["ticket_band"] = pd.cut(
    payment_behavior["order_payment_value"],
    bins=bins,
    labels=labels
)

ticket_payment_mix = pd.crosstab(
    payment_behavior["ticket_band"],
    payment_behavior["payment_type"],
    normalize="index"
) * 100

ticket_payment_mix


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

ticket_payment_mix.plot(
    kind="bar",
    stacked=True,
    ax=ax
)

ax.set_title("Composição dos meios de pagamento por faixa de ticket")
ax.set_xlabel("Faixa de ticket")
ax.set_ylabel("Participação (%)")
ax.legend(title="Meio de pagamento", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()


### Insight executivo

Essa visualização permite responder algo como:

> “O comportamento de pagamento muda conforme o valor da compra?”

Se houver concentração de cartão de crédito nas faixas de maior ticket, por exemplo, isso pode justificar uma investigação sobre **parcelamento, limite de crédito e estratégias comerciais**.

O resultado deve ser descrito como associação observada nos dados, não como relação causal.


## 12. Consolidação executiva

In [ ]:
executive_payment_summary = payment_type_summary[
    [
        "orders",
        "payment_value",
        "share_orders",
        "share_value",
        "avg_payment_record",
        "median_payment_record",
    ]
].copy()

executive_payment_summary = executive_payment_summary.rename(columns={
    "orders": "orders",
    "payment_value": "total_payment_value",
    "share_orders": "share_orders",
    "share_value": "share_financial_value",
    "avg_payment_record": "avg_payment_record_value",
    "median_payment_record": "median_payment_record_value",
})

executive_payment_summary


In [ ]:
# Indicadores gerais do universo de pagamentos
payment_kpis = pd.Series({
    "total_payment_value": payments["payment_value"].sum(),
    "payment_orders": payments["order_id"].nunique(),
    "payment_records": len(payments),
    "avg_order_payment_value": order_payment_total["order_payment_value"].mean(),
    "median_order_payment_value": order_payment_total["order_payment_value"].median(),
    "avg_payment_records_per_order": len(payments) / payments["order_id"].nunique(),
    "credit_card_share_value":
        payments.loc[
            payments["payment_type"].eq("credit_card"),
            "payment_value"
        ].sum() / payments["payment_value"].sum(),
})

payment_kpis


## 13. Regras para a narrativa executiva

Ao preparar a apresentação, priorizar quatro mensagens:

### 1. Popularidade
Qual meio possui maior participação em pedidos?

### 2. Valor
Qual meio concentra maior participação financeira?

### 3. Ticket
Quais meios estão associados aos maiores tickets?

### 4. Parcelamento
Qual é o comportamento predominante e existe associação entre maior ticket e mais parcelas?

### Exemplo de estrutura narrativa

> **“O comportamento de pagamento é concentrado em [meio]. Entretanto, a participação financeira não é necessariamente igual à participação em pedidos, indicando diferenças de ticket entre os meios. O parcelamento concentra-se em [faixa], e observamos [associação positiva/negativa/fraca] entre número de parcelas e valor do pedido. Esse comportamento deve ser considerado em estratégias de conversão, parcelamento e retenção.”**

Substitua os campos entre colchetes pelos resultados efetivamente observados.


## 14. Exportação dos resultados

As tabelas abaixo poderão ser reutilizadas no relatório e na apresentação executiva.


In [ ]:
# Exporta tabelas analíticas
payment_type_summary.to_csv(
    PROCESSED_DIR / "payment_type_summary.csv"
)

installment_summary.to_csv(
    PROCESSED_DIR / "installment_summary.csv"
)

installment_ticket.to_csv(
    PROCESSED_DIR / "installment_ticket_summary.csv"
)

ticket_payment_mix.to_csv(
    PROCESSED_DIR / "ticket_payment_mix.csv"
)

payment_behavior.to_csv(
    PROCESSED_DIR / "payment_behavior_order_level.csv",
    index=False
)

print("Resultados exportados para:", PROCESSED_DIR.resolve())


## 15. Checklist

- [x] Meios de pagamento analisados.
- [x] Participação por pedidos calculada.
- [x] Participação financeira calculada.
- [x] Ticket médio por meio analisado.
- [x] Distribuição de parcelas analisada.
- [x] Participação financeira por parcelas analisada.
- [x] Parcelamento × valor do pedido analisado.
- [x] Correlação entre parcelas e ticket calculada.
- [x] Cartão de crédito analisado separadamente.
- [x] Meio de pagamento × faixa de ticket analisado.
- [x] Tabelas exportadas para reutilização.

### Próximo notebook

**Notebook 03 — Customer RFM**

A próxima etapa será transformar o histórico de pedidos em uma visão de cliente usando `customer_unique_id`, calculando **Recency, Frequency e Monetary**, criando segmentos e identificando quais grupos concentram o maior valor econômico.
